# ***Environment Setup***





In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install "opacus<1.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 5.1 MB/s eta 0:00:00


In [3]:
!pip install -U torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/5

In [4]:
!pip install -U synthcity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 16.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of monai to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of shap to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tsai to determine which version is compatible with other requir

# ***Adult Dataset***

## ***Data Preprocissing***

In [ ]:
"""Preprocessing pipeline for the UCI Adult dataset.

This script loads separate training and test files, performs Adult-specific
cleaning (missing-value markers, redundant column removal, and test-label dot
cleanup), and applies leakage-safe preprocessing by fitting all transformers on
the training set only. Techniques used include most-frequent imputation,
one-hot encoding for categorical features, MinMax scaling for numeric features,
and binary target encoding, then saving processed train/test CSV outputs.
"""


import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder


# File paths
TRAIN_PATH = r"/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_before/adult.data"
TEST_PATH = r"/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_before/adult.test"
OUTPUT_DIR = r"/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_after"


# UCI Adult column names
COLUMN_NAMES = [
	"age",
	"workclass",
	"fnlwgt",
	"education",
	"education-num",
	"marital-status",
	"occupation",
	"relationship",
	"race",
	"sex",
	"capital-gain",
	"capital-loss",
	"hours-per-week",
	"native-country",
	"salary",
]


def main() -> None:
	# Load train/test with explicit column names from UCI schema.
	df_train = pd.read_csv(
		TRAIN_PATH,
		header=None,
		names=COLUMN_NAMES,
		na_values=" ?",
		skipinitialspace=False,
	)

	# adult.test includes a first metadata row to be skipped.
	df_test = pd.read_csv(
		TEST_PATH,
		header=None,
		names=COLUMN_NAMES,
		na_values=" ?",
		skiprows=1,
		skipinitialspace=False,
	)

	# Initial cleaning
	df_train = df_train.drop(columns=["education-num"])
	df_test = df_test.drop(columns=["education-num"])

	# Remove trailing period in test labels (<=50K. / >50K.) to match train labels.
	df_test["salary"] = df_test["salary"].str.replace(".", "", regex=False)

	# Split features/target
	X_train = df_train.drop(columns=["salary"])
	y_train = df_train["salary"]

	X_test = df_test.drop(columns=["salary"])
	y_test = df_test["salary"]

	# Identify feature types
	categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
	numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

	# Missing value imputation (fit on train only)
	cat_imputer = SimpleImputer(strategy="most_frequent")
	num_imputer = SimpleImputer(strategy="most_frequent")

	X_train_cat = cat_imputer.fit_transform(X_train[categorical_cols])
	X_test_cat = cat_imputer.transform(X_test[categorical_cols])

	X_train_num = num_imputer.fit_transform(X_train[numeric_cols])
	X_test_num = num_imputer.transform(X_test[numeric_cols])

	# Categorical encoding (fit on train only)
	ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
	X_train_cat_ohe = ohe.fit_transform(X_train_cat)
	X_test_cat_ohe = ohe.transform(X_test_cat)

	ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
	X_train_cat_df = pd.DataFrame(X_train_cat_ohe, columns=ohe_feature_names, index=X_train.index)
	X_test_cat_df = pd.DataFrame(X_test_cat_ohe, columns=ohe_feature_names, index=X_test.index)

	# Ensure column names are strings
	X_train_cat_df.columns = X_train_cat_df.columns.astype(str)
	X_test_cat_df.columns = X_test_cat_df.columns.astype(str)

	# Numeric scaling to [0, 1] (fit on train only)
	scaler = MinMaxScaler()
	X_train_num_scaled = scaler.fit_transform(X_train_num)
	X_test_num_scaled = scaler.transform(X_test_num)

	X_train_num_df = pd.DataFrame(X_train_num_scaled, columns=numeric_cols, index=X_train.index)
	X_test_num_df = pd.DataFrame(X_test_num_scaled, columns=numeric_cols, index=X_test.index)

	# Combine numeric + encoded categorical features
	X_train_processed = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
	X_test_processed = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

	# Target encoding (fit on train only)
	target_encoder = LabelEncoder()
	y_train_encoded = target_encoder.fit_transform(y_train)
	y_test_encoded = target_encoder.transform(y_test)

	train_processed = X_train_processed.copy()
	test_processed = X_test_processed.copy()
	train_processed["salary"] = y_train_encoded
	test_processed["salary"] = y_test_encoded

	os.makedirs(OUTPUT_DIR, exist_ok=True)
	train_out = os.path.join(OUTPUT_DIR, "adult_train_preprocessed.csv")
	test_out = os.path.join(OUTPUT_DIR, "adult_test_preprocessed.csv")

	train_processed.to_csv(train_out, index=False)
	test_processed.to_csv(test_out, index=False)

	print(f"Saved preprocessed training data to: {train_out}")
	print(f"Saved preprocessed test data to: {test_out}")


if __name__ == "__main__":
	main()


## ***PATE-GAN Training***

In [ ]:
"""
Train PATE-GAN with Synthcity on tabular data with:
- automatic resume from latest checkpoint
- periodic checkpointing
- epoch-level CSV logging (generator/discriminator losses + privacy spent)
- final synthetic data export
- final model export

"""

from __future__ import annotations

import argparse
import logging
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch



try:
    # Some Synthcity versions may expose this path.
    from synthcity.plugins.generic.plugin_pategan import PATEGANPlugin, Teachers
except ImportError:
    # Current Synthcity versions expose PATE-GAN under privacy plugins.
    from synthcity.plugins.privacy.plugin_pategan import PATEGANPlugin, Teachers

from synthcity.plugins.core.models.tabular_gan import TabularGAN



# User placeholders
TRAIN_DATA_PATH = "/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_after/adult_train_preprocessed.csv"
OUTPUT_DIR = "/content/drive/MyDrive/PATE-GAN"

# Random seed to run index mapping used for automatic output sub-folders.
RUN_MAPPING = {
    42: 1,
    13: 2,
    101: 3,
    1234: 4,
    2026: 5,
}

# Core training hyperparameters
N_ITER = 200
BATCH_SIZE = 128
EPSILON = 1.0
N_TEACHERS = 100
TARGET_COLUMN = "salary"

# Checkpoint settings
CHECKPOINT_EVERY = 30

# Misc
RANDOM_STATE = 1234
DEVICE = "auto"  # "auto", "cpu", "cuda"
SYNTHETIC_ROWS = None  # None -> same as training set size


@dataclass
class RunState:
    plugin: Any
    current_iter: int
    epsilon_hat: float
    history: List[Dict[str, Any]]
    train_columns: List[str]


def resolve_device(device: str) -> str:
    if device == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if device not in {"cpu", "cuda"}:
        raise ValueError("DEVICE must be one of: auto, cpu, cuda")
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("DEVICE is set to 'cuda' but CUDA is not available.")
    return device


def resolve_run_name(random_state: int) -> str:
    run_number = RUN_MAPPING.get(int(random_state))
    if run_number is not None:
        return f"Run_{run_number}"
    return f"Run_seed_{int(random_state)}"


def setup_logging(output_dir: Path) -> logging.Logger:
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file = output_dir / "training.log"

    logger = logging.getLogger("pategan_train")
    logger.setLevel(logging.INFO)

    # Close previous handlers to avoid file descriptor leaks when reinitializing per run.
    for handler in list(logger.handlers):
        try:
            handler.close()
        except Exception:
            pass
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)
    logger.addHandler(file_handler)
    return logger


def setup_batch_logger() -> logging.Logger:
    logger = logging.getLogger("pategan_batch")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    return logger


def load_training_data(train_data_path: Path) -> pd.DataFrame:
    if not train_data_path.exists():
        raise FileNotFoundError(f"Training file not found: {train_data_path}")

    df = pd.read_csv(train_data_path)
    if df.empty:
        raise ValueError("Training CSV is empty.")
    return df


def checkpoint_path(output_dir: Path, iteration: int) -> Path:
    return output_dir / f"checkpoint_iter_{iteration:05d}.joblib"


def find_latest_checkpoint(output_dir: Path) -> Optional[Path]:
    checkpoints = sorted(output_dir.glob("checkpoint_iter_*.joblib"))
    if not checkpoints:
        return None
    return checkpoints[-1]


def list_checkpoints_desc(output_dir: Path) -> List[Path]:
    return sorted(output_dir.glob("checkpoint_iter_*.joblib"), reverse=True)


def save_checkpoint(output_dir: Path, state: RunState) -> Path:
    ckpt = checkpoint_path(output_dir, state.current_iter)
    payload = {
        "plugin": state.plugin,
        "current_iter": state.current_iter,
        "epsilon_hat": state.epsilon_hat,
        "history": state.history,
        "train_columns": state.train_columns,
        "resumable": True,
    }

    try:
        joblib.dump(payload, ckpt)
    except Exception:
        # Some Colab/Synthcity builds contain dynamically generated classes
        # that cannot be pickled. Save a lightweight checkpoint instead.
        lightweight_payload = {
            "plugin": None,
            "current_iter": state.current_iter,
            "epsilon_hat": state.epsilon_hat,
            "history": state.history,
            "train_columns": state.train_columns,
            "resumable": False,
        }
        joblib.dump(lightweight_payload, ckpt)
    return ckpt


def load_checkpoint(checkpoint_file: Path) -> RunState:
    payload = joblib.load(checkpoint_file)
    if payload.get("plugin") is None:
        raise RuntimeError(
            "Checkpoint is metadata-only and cannot resume model state. "
            "Run with --force-restart to start a new training run."
        )

    return RunState(
        plugin=payload["plugin"],
        current_iter=int(payload["current_iter"]),
        epsilon_hat=float(payload["epsilon_hat"]),
        history=list(payload["history"]),
        train_columns=list(payload["train_columns"]),
    )


def build_plugin(
    n_iter: int,
    batch_size: int,
    epsilon: float,
    n_teachers: int,
    random_state: int,
    device: str,
    workspace: Path,
) -> Any:
    plugin = PATEGANPlugin(
        n_iter=n_iter,
        batch_size=batch_size,
        epsilon=epsilon,
        n_teachers=n_teachers,
        random_state=random_state,
        device=device,
        workspace=workspace,
    )
    return plugin


def initialize_manual_training_state(plugin: Any, train_df: pd.DataFrame) -> Tuple[pd.DataFrame, float]:
    """
    Reproduces the Synthcity PATEGAN.fit initialization so we can checkpoint/resume
    every outer iteration while preserving native behavior.
    """
    pategan = plugin.model

    pategan.columns = train_df.columns
    if pategan.delta is None:
        pategan.delta = 1 / (len(train_df) * np.sqrt(len(train_df)))

    pategan.model = TabularGAN(
        train_df,
        n_units_latent=pategan.generator_n_units_hidden,
        batch_size=pategan.batch_size,
        generator_n_layers_hidden=pategan.generator_n_layers_hidden,
        generator_n_units_hidden=pategan.generator_n_units_hidden,
        generator_nonlin=pategan.generator_nonlin,
        generator_nonlin_out_discrete="softmax",
        generator_nonlin_out_continuous="none",
        generator_lr=pategan.lr,
        generator_residual=True,
        generator_n_iter=pategan.generator_n_iter,
        generator_batch_norm=False,
        generator_dropout=0,
        generator_weight_decay=pategan.weight_decay,
        discriminator_n_units_hidden=pategan.discriminator_n_units_hidden,
        discriminator_n_layers_hidden=pategan.discriminator_n_layers_hidden,
        discriminator_n_iter=pategan.discriminator_n_iter,
        discriminator_nonlin=pategan.discriminator_nonlin,
        discriminator_batch_norm=False,
        discriminator_dropout=pategan.discriminator_dropout,
        discriminator_lr=pategan.lr,
        discriminator_weight_decay=pategan.weight_decay,
        clipping_value=pategan.clipping_value,
        encoder_max_clusters=pategan.encoder_max_clusters,
        encoder=pategan.encoder,
        n_iter_print=max(1, pategan.generator_n_iter - 1),
        device=pategan.device,
    )

    x_train_enc = pategan.model.encode(train_df)
    pategan.samples_per_teacher = max(1, int(len(x_train_enc) / max(1, pategan.n_teachers)))
    pategan.alpha_dict = np.zeros([pategan.alpha])

    epsilon_hat = 0.0
    return x_train_enc, epsilon_hat


def run_one_outer_iteration(
    plugin: Any,
    x_train_enc: pd.DataFrame,
    epsilon_hat: float,
) -> Tuple[float, float, float, int]:
    """
    Executes one outer PATE iteration:
    1) train teachers
    2) train student GAN
    3) update privacy accountant

    Returns:
        generator_loss_mean, discriminator_loss_mean, epsilon_hat, inner_steps
    """
    pategan = plugin.model

    teachers = Teachers(
        n_teachers=pategan.n_teachers,
        samples_per_teacher=pategan.samples_per_teacher,
        lamda=pategan.lamda,
        template=pategan.teacher_template,
    )
    teachers.fit(np.asarray(x_train_enc), pategan.model)

    def fake_labels_generator(x_batch: torch.Tensor) -> torch.Tensor:
        if pategan.n_teachers == 0:
            return torch.zeros((len(x_batch),))

        x_df = pd.DataFrame(x_batch.detach().cpu().numpy())
        n0_mb, n1_mb, y_mb = teachers.pate_lamda(np.asarray(x_df))

        if np.sum(y_mb) >= len(x_batch) / 2:
            return torch.zeros((len(x_batch),))

        pategan._update_alpha(n0_mb, n1_mb)
        return torch.from_numpy(np.reshape(np.asarray(y_mb, dtype=int), [-1, 1]))

    # Capture GAN epoch losses by wrapping internal train epoch.
    inner_losses: List[Tuple[float, float]] = []
    base_gan = pategan.model.model
    original_train_epoch = base_gan._train_epoch

    def wrapped_train_epoch(*args: Any, **kwargs: Any) -> Tuple[float, float]:
        g_loss, d_loss = original_train_epoch(*args, **kwargs)
        inner_losses.append((float(g_loss), float(d_loss)))
        return g_loss, d_loss

    base_gan._train_epoch = wrapped_train_epoch
    try:
        pategan.model.fit(
            x_train_enc,
            fake_labels_generator=fake_labels_generator,
            encoded=True,
        )
    finally:
        base_gan._train_epoch = original_train_epoch

    curr_list: List[float] = []
    for lidx in range(pategan.alpha):
        local_alpha = (pategan.alpha_dict[lidx] - np.log(pategan.delta)) / float(lidx + 1)
        curr_list.append(float(local_alpha))

    epsilon_hat = float(np.min(curr_list))

    if inner_losses:
        g_loss_mean = float(np.mean([x[0] for x in inner_losses]))
        d_loss_mean = float(np.mean([x[1] for x in inner_losses]))
    else:
        g_loss_mean = float("nan")
        d_loss_mean = float("nan")

    return g_loss_mean, d_loss_mean, epsilon_hat, len(inner_losses)


def write_history_csv(history: List[Dict[str, Any]], output_dir: Path) -> Path:
    history_path = output_dir / "training_history.csv"
    pd.DataFrame(history).to_csv(history_path, index=False)
    return history_path


def generate_synthetic_dataframe(plugin: Any, count: int) -> pd.DataFrame:
    syn: Any = None
    errors: List[str] = []

    # Try plugin API first.
    try:
        syn = plugin.generate(count=count)
    except Exception as e:
        errors.append(f"plugin.generate(count=...): {e}")

    # If plugin wrapper reports not fitted, call implementation methods directly.
    if syn is None and hasattr(plugin, "_generate"):
        try:
            syn = plugin._generate(count=count)
        except Exception as e:
            errors.append(f"plugin._generate(count=...): {e}")

    # Compatibility fallbacks across Synthcity internals.
    if syn is None and hasattr(plugin, "model"):
        pmodel = plugin.model

        if hasattr(pmodel, "generate"):
            try:
                syn = pmodel.generate(count=count)
            except Exception as e:
                errors.append(f"plugin.model.generate(count=...): {e}")

        if syn is None and hasattr(pmodel, "_generate"):
            try:
                syn = pmodel._generate(count=count)
            except Exception as e:
                errors.append(f"plugin.model._generate(count=...): {e}")

        # Nested GAN object used by some versions.
        if syn is None and hasattr(pmodel, "model"):
            inner = pmodel.model

            if hasattr(inner, "generate"):
                try:
                    syn = inner.generate(count=count)
                except Exception as e:
                    errors.append(f"plugin.model.model.generate(count=...): {e}")

            if syn is None and hasattr(inner, "sample"):
                try:
                    syn = inner.sample(count)
                except Exception as e:
                    errors.append(f"plugin.model.model.sample(count): {e}")

    if syn is None:
        raise RuntimeError(
            "Unable to generate synthetic data with available Synthcity APIs. "
            f"Tried methods and got: {' | '.join(errors)}"
        )

    if isinstance(syn, tuple) and len(syn) > 0:
        syn = syn[0]

    # Compatibility across synthcity versions:
    # - some return DataLoader-like objects with dataframe()
    # - some return DataFrame directly
    if hasattr(syn, "dataframe"):
        return syn.dataframe()
    if isinstance(syn, pd.DataFrame):
        return syn
    if isinstance(syn, np.ndarray):
        return pd.DataFrame(syn)
    if torch.is_tensor(syn):
        return pd.DataFrame(syn.detach().cpu().numpy())

    raise TypeError(
        "Unsupported output from synthetic generation. "
        f"Type received: {type(syn)}"
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train Synthcity PATE-GAN with checkpoints and logs")
    parser.add_argument("--train-data-path", type=str, default=TRAIN_DATA_PATH)
    parser.add_argument("--output-dir", type=str, default=OUTPUT_DIR)
    parser.add_argument("--target-column", type=str, default=TARGET_COLUMN)
    parser.add_argument("--n-iter", type=int, default=N_ITER)
    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    parser.add_argument("--epsilon", type=float, default=EPSILON)
    parser.add_argument("--n-teachers", type=int, default=N_TEACHERS)
    parser.add_argument("--checkpoint-every", type=int, default=CHECKPOINT_EVERY)
    parser.add_argument("--random-state", type=int, default=RANDOM_STATE)
    parser.add_argument("--device", type=str, default=DEVICE)
    parser.add_argument("--synthetic-rows", type=int, default=SYNTHETIC_ROWS)
    parser.add_argument(
        "--force-restart",
        action="store_true",
        help="Ignore available checkpoints and start training from scratch.",
    )
    # In notebooks (e.g., Colab), Jupyter injects args like "-f <kernel.json>".
    # parse_known_args keeps CLI behavior while safely ignoring those extras.
    args, _ = parser.parse_known_args()
    return args


def run_is_complete(output_dir: Path, target_epsilon: float, target_n_iter: int) -> bool:
    history_csv = output_dir / "training_history.csv"
    synthetic_csv = output_dir / "synthetic_data.csv"
    if not history_csv.exists() or not synthetic_csv.exists():
        return False

    try:
        history_df = pd.read_csv(history_csv)
        if history_df.empty:
            return False

        last_row = history_df.iloc[-1]
        last_epsilon = float(last_row.get("epsilon_hat", float("nan")))
        last_iter = int(last_row.get("iter", 0))
        return bool(last_epsilon >= target_epsilon or last_iter >= target_n_iter)
    except Exception:
        return False


def run_single_experiment(
    args: argparse.Namespace,
    train_df: pd.DataFrame,
    device: str,
    random_state: int,
) -> str:
    run_name = resolve_run_name(random_state)
    output_dir = Path(args.output_dir) / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(output_dir)
    logger.info("=" * 80)
    logger.info("Starting run: %s", run_name)
    logger.info("Random state: %d", int(random_state))
    logger.info("Output directory: %s", output_dir)

    if not args.force_restart and run_is_complete(output_dir, float(args.epsilon), int(args.n_iter)):
        logger.info(
            "Run already completed (history + synthetic data found and stopping criterion met). "
            "Skipping this run."
        )
        return "skipped"

    latest_ckpt: Optional[Path] = None
    state: Optional[RunState] = None

    if not args.force_restart:
        for ckpt in list_checkpoints_desc(output_dir):
            logger.info("Trying checkpoint: %s", ckpt)
            try:
                state = load_checkpoint(ckpt)
                latest_ckpt = ckpt
                logger.info("Resuming from checkpoint: %s", latest_ckpt)
                break
            except Exception as e:
                logger.warning("Skipping checkpoint %s due to load error: %s", ckpt, e)

    if latest_ckpt is not None and state is not None:
        if state.train_columns != list(train_df.columns):
            raise ValueError(
                "Training columns differ from checkpoint columns. "
                "Cannot safely resume."
            )

        plugin = state.plugin
        current_iter = state.current_iter
        epsilon_hat = state.epsilon_hat
        history = state.history

        if not hasattr(plugin.model, "model"):
            raise RuntimeError(
                "Checkpoint does not include an initialized TabularGAN state. "
                "Please restart with --force-restart."
            )

        # Keep the latest command line budget as an upper bound.
        plugin.model.max_iter = int(args.n_iter)
        plugin.model.epsilon = float(args.epsilon)
        plugin.model.batch_size = int(args.batch_size)
        plugin.model.n_teachers = int(args.n_teachers)

        x_train_enc = plugin.model.model.encode(train_df)
        plugin.model.samples_per_teacher = max(
            1,
            int(len(x_train_enc) / max(1, plugin.model.n_teachers)),
        )
    else:
        logger.info("No checkpoint found. Starting a fresh training run.")
        plugin = build_plugin(
            n_iter=int(args.n_iter),
            batch_size=int(args.batch_size),
            epsilon=float(args.epsilon),
            n_teachers=int(args.n_teachers),
            random_state=int(random_state),
            device=device,
            workspace=output_dir / "workspace",
        )
        x_train_enc, epsilon_hat = initialize_manual_training_state(plugin, train_df)
        current_iter = 0
        history = []

    # Main outer training loop (privacy iterations)
    logger.info(
        "Starting training loop: target max_iter=%d, epsilon=%.6f, checkpoint_every=%d",
        int(args.n_iter),
        float(args.epsilon),
        int(args.checkpoint_every),
    )

    while epsilon_hat < float(args.epsilon) and current_iter < int(args.n_iter):
        current_iter += 1
        t0 = time.time()

        g_loss, d_loss, epsilon_hat, inner_steps = run_one_outer_iteration(
            plugin=plugin,
            x_train_enc=x_train_enc,
            epsilon_hat=epsilon_hat,
        )

        elapsed = time.time() - t0

        row = {
            "iter": current_iter,
            "generator_loss": g_loss,
            "discriminator_loss": d_loss,
            "epsilon_hat": epsilon_hat,
            "epsilon_target": float(args.epsilon),
            "delta": float(plugin.model.delta),
            "privacy_spent": epsilon_hat,
            "inner_gan_steps": inner_steps,
            "elapsed_seconds": elapsed,
        }
        history.append(row)

        logger.info(
            "Iter %d | G_loss=%.6f | D_loss=%.6f | epsilon_hat=%.6f/%.6f | inner_steps=%d | %.2fs",
            current_iter,
            g_loss,
            d_loss,
            epsilon_hat,
            float(args.epsilon),
            inner_steps,
            elapsed,
        )

        if current_iter % int(args.checkpoint_every) == 0:
            ckpt_file = save_checkpoint(
                output_dir,
                RunState(
                    plugin=plugin,
                    current_iter=current_iter,
                    epsilon_hat=epsilon_hat,
                    history=history,
                    train_columns=list(train_df.columns),
                ),
            )
            logger.info("Checkpoint saved: %s", ckpt_file)

    # Always save final checkpoint
    final_ckpt = save_checkpoint(
        output_dir,
        RunState(
            plugin=plugin,
            current_iter=current_iter,
            epsilon_hat=epsilon_hat,
            history=history,
            train_columns=list(train_df.columns),
        ),
    )
    logger.info("Final checkpoint saved: %s", final_ckpt)

    # Persist final model separately
    final_model_path = output_dir / "pategan_final_model.joblib"
    try:
        joblib.dump(plugin, final_model_path)
        logger.info("Final model saved: %s", final_model_path)
    except Exception as e:
        logger.warning(
            "Final model could not be pickled in this environment (%s). "
            "Training outputs are still saved (history CSV and synthetic data).",
            e,
        )

    # Persist history CSV
    history_csv_path = write_history_csv(history, output_dir)
    logger.info("Training history saved: %s", history_csv_path)

    # Manual training bypasses plugin.fit(), so mark wrapper as fitted
    # to keep generation pathways compatible across Synthcity versions.
    if hasattr(plugin, "fitted"):
        plugin.fitted = True

    # Generate synthetic data
    n_rows = len(train_df) if args.synthetic_rows is None else int(args.synthetic_rows)
    synthetic_df = generate_synthetic_dataframe(plugin, count=n_rows)

    # Ensure same columns ordering if available.
    if list(train_df.columns) == list(synthetic_df.columns):
        synthetic_df = synthetic_df[train_df.columns]

    synthetic_path = output_dir / "synthetic_data.csv"
    synthetic_df.to_csv(synthetic_path, index=False)
    logger.info("Synthetic data saved: %s | shape=%s", synthetic_path, synthetic_df.shape)
    logger.info("Run completed successfully: %s", run_name)
    return "completed"


def main() -> None:
    args = parse_args()

    if not args.train_data_path:
        raise ValueError("Please set TRAIN_DATA_PATH or pass --train-data-path.")
    if not args.output_dir:
        raise ValueError("Please set OUTPUT_DIR or pass --output-dir.")
    if int(args.n_teachers) < 0:
        raise ValueError("--n-teachers must be >= 0.")

    device = resolve_device(args.device)
    batch_logger = setup_batch_logger()

    batch_logger.info("Loading training data...")
    train_df = load_training_data(Path(args.train_data_path))

    if args.target_column:
        if args.target_column not in train_df.columns:
            raise ValueError(
                f"Target column '{args.target_column}' was not found in the training data. "
                f"Available columns: {list(train_df.columns)}"
            )
    else:
        raise ValueError("--target-column cannot be empty.")

    batch_logger.info("Device: %s", device)
    batch_logger.info("Train shape: %s", train_df.shape)
    batch_logger.info("Target column: %s", args.target_column)
    batch_logger.info("Number of teachers: %d", int(args.n_teachers))
    batch_logger.info("Base output directory: %s", Path(args.output_dir))
    batch_logger.info("Starting sequential multi-run training for all mapped seeds.")

    seeds: List[int] = list(RUN_MAPPING.keys())
    completed = 0
    skipped = 0
    failed = 0

    for seed in seeds:
        run_name = resolve_run_name(seed)
        batch_logger.info("\n--- [%s] seed=%d ---", run_name, seed)
        try:
            status = run_single_experiment(
                args=args,
                train_df=train_df,
                device=device,
                random_state=int(seed),
            )

            if status == "completed":
                completed += 1
            elif status == "skipped":
                skipped += 1
            else:
                failed += 1
        except Exception as e:
            failed += 1
            batch_logger.error("Run failed for seed=%d (%s): %s", seed, run_name, e)
            batch_logger.error("Traceback:\n%s", traceback.format_exc())
            continue

    batch_logger.info(
        "\nAll runs processed. Summary -> completed=%d, skipped=%d, failed=%d, total=%d",
        completed,
        skipped,
        failed,
        len(seeds),
    )


if __name__ == "__main__":
    main()


## ***Classifier Fine-Tuning & HPO (RandomizedSearchCV)***

In [ ]:
"""
Hyperparameter Optimization for Adult Dataset (Classification)

Models:
1. Logistic Regression
2. AdaBoost
3. KNN
4. XGBoost

Search Method:
- RandomizedSearchCV
- 5-fold Stratified Cross-Validation

Optimization Metric:
- AUCPR (average_precision)

Input file path (default for Google Colab):
/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_after/adult_train_preprocessed.csv
"""

from __future__ import annotations

import os
import sys
import subprocess
import warnings
from pprint import pformat

import pandas as pd
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")



# User Configuration

TRAIN_PATH = "/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_after/adult_train_preprocessed.csv"
TARGET_COL = "salary"

OPTIMIZE_METRIC = "average_precision"

RANDOM_STATE = 42
CV_SPLITS = 5
N_JOBS = 2
N_ITER_DEFAULT = 15
N_ITER_XGB = 20
VERBOSE = 2



# Helpers

def ensure_xgboost_installed() -> None:
    """Install xgboost in the current environment if missing."""
    try:
        import xgboost  # noqa: F401
    except ImportError:
        print("xgboost not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])


def load_dataset(train_path: str, target_col: str):
    """Load data and return (X, y) with encoded binary target if needed."""
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Training file not found at: {train_path}")

    df = pd.read_csv(train_path)

    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found. Available columns: {list(df.columns)}"
        )

    X = df.drop(columns=[target_col])
    y = df[target_col]

    if y.dtype == "object" or str(y.dtype).startswith("category"):
        encoder = LabelEncoder()
        y = encoder.fit_transform(y)
    else:
        y = y.astype(int)

    return X, y


def build_adaboost(random_state: int) -> AdaBoostClassifier:
    """
    Build AdaBoost with depth-2 decision tree base estimator.
    Keeps compatibility with sklearn versions using either
    'estimator' (new) or 'base_estimator' (older).
    """
    base_tree = DecisionTreeClassifier(max_depth=2, random_state=random_state)
    try:
        return AdaBoostClassifier(estimator=base_tree, random_state=random_state)
    except TypeError:
        return AdaBoostClassifier(base_estimator=base_tree, random_state=random_state)


def run_hpo(X, y) -> dict:
    """Run RandomizedSearchCV for all requested models and return result dictionary."""
    ensure_xgboost_installed()
    from xgboost import XGBClassifier

    # Force AUCPR as requested for imbalanced classification quality.
    scoring = "average_precision"
    scoring_name = "AUCPR (Average Precision)"

    cv = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    models_and_spaces = {
        "Logistic Regression": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(
                    solver="liblinear",  # faster than saga on this search setup
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                )),
            ]),
            "param_distributions": {
                "model__C": loguniform(1e-3, 1e2),
                "model__penalty": ["l1", "l2"],
            },
            "n_iter": N_ITER_DEFAULT,
        },
        "AdaBoost": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("model", build_adaboost(RANDOM_STATE)),
            ]),
            "param_distributions": {
                "model__n_estimators": randint(50, 501),
                "model__learning_rate": loguniform(1e-3, 1.0),
            },
            "n_iter": N_ITER_DEFAULT,
        },
        "KNN": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier()),
            ]),
            "param_distributions": {
                "model__n_neighbors": [3, 5, 7, 9],
                "model__weights": ["uniform", "distance"],
            },
            # Full space size = 8 combinations. Keeping randomized API as requested.
            "n_iter": 8,
        },
        "XGBoost": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("model", XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS,
                    tree_method="hist",
                    device="cuda",
                )),
            ]),
            "param_distributions": {
                "model__n_estimators": randint(100, 601),
                "model__max_depth": randint(3, 11),
                "model__learning_rate": loguniform(1e-3, 0.3),
                "model__subsample": uniform(0.6, 0.4),
            },
            "n_iter": N_ITER_XGB,
        },
    }

    print("=" * 90)
    print("Hyperparameter Optimization Started")
    print(f"Optimization metric: {scoring_name}")
    print(
        f"CV strategy: StratifiedKFold(n_splits={CV_SPLITS}, shuffle=True, random_state={RANDOM_STATE})"
    )
    print("=" * 90)

    results = {}

    for model_name, cfg in models_and_spaces.items():
        print(f"\n>>> Optimizing: {model_name}")

        search = RandomizedSearchCV(
            estimator=cfg["pipeline"],
            param_distributions=cfg["param_distributions"],
            n_iter=cfg["n_iter"],
            scoring=scoring,
            cv=cv,
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
            refit=True,
        )

        search.fit(X, y)

        best_params = search.best_params_
        best_score = search.best_score_

        results[model_name] = {
            "best_params": best_params,
            "best_score": best_score,
        }

        print("-" * 90)
        print(f"{model_name} - BEST RESULTS")
        print(f"Best {scoring_name}: {best_score:.6f}")
        print("Best Parameters:")
        print(pformat(best_params, sort_dicts=True))
        print("-" * 90)

    print("\n" + "=" * 90)
    print("FINAL SUMMARY")
    for model_name, result in results.items():
        print(f"{model_name}: {scoring_name} = {result['best_score']:.6f}")

    best_params_by_model = {
        model_name: result["best_params"] for model_name, result in results.items()
    }
    print("\nBEST PARAMETERS DICTIONARY (copy-ready):")
    print(pformat(best_params_by_model, sort_dicts=True))
    print("=" * 90)

    return results


def main() -> None:
    X, y = load_dataset(TRAIN_PATH, TARGET_COL)
    run_hpo(X, y)


if __name__ == "__main__":
    main()


Hyperparameter Optimization Started
Optimization metric: AUCPR (Average Precision)
CV strategy: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

>>> Optimizing: Logistic Regression
Fitting 5 folds for each of 15 candidates, totalling 75 fits
------------------------------------------------------------------------------------------
Logistic Regression - BEST RESULTS
Best AUCPR (Average Precision): 0.764355
Best Parameters:
{'model__C': np.float64(0.0745934328572655), 'model__penalty': 'l1'}
------------------------------------------------------------------------------------------

>>> Optimizing: AdaBoost
Fitting 5 folds for each of 15 candidates, totalling 75 fits
------------------------------------------------------------------------------------------
AdaBoost - BEST RESULTS
Best AUCPR (Average Precision): 0.792917
Best Parameters:
{'model__learning_rate': np.float64(0.31428808908401085),
 'model__n_estimators': 343}
--------------------------------------------------------

## ***Utility Evaluation***

In [ ]:

"""Utility evaluation pipeline for the preprocessed Adult dataset.

This script trains and compares four binary classifiers (Logistic Regression,
AdaBoost, KNN, and XGBoost), uses 10-fold Stratified
Cross-Validation on the training set, and evaluates final performance on a
held-out test set using Accuracy, Recall, F1-Score, AUROC, and AUCPR.
It exports summary tables and confusion matrices to the results directory.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
	accuracy_score,
	average_precision_score,
	confusion_matrix,
	f1_score,
	recall_score,
	roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


TEST_PATH = Path(
	r"/content/drive/MyDrive/PATE-TransGAN/Adult/Adult_after/adult_test_preprocessed.csv"
)
BASE_RUN_DIR = Path(r"/content/drive/MyDrive/PATE-GAN")
RUN_IDS = [1, 2, 3, 4, 5]
TARGET_COLUMN = "salary"


def load_data(train_path: Path, test_path: Path, target_column: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
	"""Load train/test CSVs and split features/target."""
	if not train_path.exists():
		raise FileNotFoundError(f"Training file not found: {train_path}")
	if not test_path.exists():
		raise FileNotFoundError(f"Test file not found: {test_path}")

	train_df = pd.read_csv(train_path)
	test_df = pd.read_csv(test_path)

	if target_column not in train_df.columns:
		raise KeyError(f"Target column '{target_column}' is missing from training data.")
	if target_column not in test_df.columns:
		raise KeyError(f"Target column '{target_column}' is missing from test data.")

	x_train = train_df.drop(columns=[target_column])
	y_train = train_df[target_column].copy()
	x_test = test_df.drop(columns=[target_column])
	y_test = test_df[target_column].copy()

	return x_train, y_train, x_test, y_test


def encode_target(y_train: pd.Series, y_test: pd.Series) -> Tuple[pd.Series, pd.Series, LabelEncoder]:
	"""Encode target labels so metrics are computed consistently."""
	label_encoder = LabelEncoder()
	y_train_encoded = pd.Series(label_encoder.fit_transform(y_train), index=y_train.index)

	unseen_labels = set(y_test.unique()) - set(label_encoder.classes_)
	if unseen_labels:
		raise ValueError(
			f"Test target contains unseen labels not present in training set: {sorted(unseen_labels)}"
		)

	y_test_encoded = pd.Series(label_encoder.transform(y_test), index=y_test.index)

	if len(label_encoder.classes_) != 2:
		raise ValueError(
			f"Expected a binary target for requested metrics, but found {len(label_encoder.classes_)} classes."
		)

	return y_train_encoded, y_test_encoded, label_encoder


def build_models() -> Dict[str, Pipeline]:
	"""Create all requested classification models."""
	scaler = ColumnTransformer(
		transformers=[("num", StandardScaler(), slice(0, None))],
		remainder="drop",
	)

	models: Dict[str, Pipeline] = {
		"Logistic Regression": Pipeline(
			steps=[
				("scaler", scaler),
				(
					"classifier",
					LogisticRegression(
						C=0.0745934328572655,
						penalty="l1",
						solver="liblinear",
						max_iter=1000,
						random_state=42,
					),
				),
			]
		),
		"AdaBoost": Pipeline(
			steps=[
				(
					"classifier",
					AdaBoostClassifier(
						estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
						n_estimators=343,
						learning_rate=0.31428808908401085,
						random_state=42,
					),
				),
			]
		),
		"KNN": Pipeline(
			steps=[
				("scaler", scaler),
				("classifier", KNeighborsClassifier(n_neighbors=9, weights="distance")),
			]
		),
		"XGBoost": Pipeline(
			steps=[
				(
					"classifier",
					XGBClassifier(
						objective="binary:logistic",
						eval_metric="logloss",
						n_estimators=363,
						learning_rate=0.04926364988526881,
						max_depth=6,
						subsample=0.6137554084460873,
						colsample_bytree=0.9,
						random_state=42,
						n_jobs=-1,
						tree_method="hist",
						device="cuda",
					),
				)
			]
		),
	}

	return models


def evaluate_models(
	models: Dict[str, Pipeline],
	x_train: pd.DataFrame,
	y_train: pd.Series,
	x_test: pd.DataFrame,
	y_test: pd.Series,
):
	"""Run CV on train data, then fit on full train and evaluate on test data."""
	cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
	cv_scoring = {
		"accuracy": "accuracy",
		"recall": "recall",
		"f1": "f1",
		"roc_auc": "roc_auc",
		"auc_pr": "average_precision",
	}

	cv_rows = []
	auc_rows = []
	metrics_rows = []
	confusion_details = []

	for model_name, model in models.items():
		print(f"Training and Evaluating: {model_name}...")
		cv_results = cross_validate(
			model,
			x_train,
			y_train,
			cv=cv,
			scoring=cv_scoring,
			n_jobs=-1,
			error_score="raise",
		)

		cv_row = {
			"Model Name": model_name,
			"CV Accuracy": cv_results["test_accuracy"].mean(),
			"CV Recall": cv_results["test_recall"].mean(),
			"CV F1-Score": cv_results["test_f1"].mean(),
			"CV AUROC": cv_results["test_roc_auc"].mean(),
			"CV AUCPR": cv_results["test_auc_pr"].mean(),
		}
		cv_rows.append(cv_row)

		model.fit(x_train, y_train)
		y_pred = model.predict(x_test)

		if not hasattr(model, "predict_proba"):
			raise AttributeError(f"Model '{model_name}' does not support predict_proba, required for AUROC/AUCPR.")
		y_proba = model.predict_proba(x_test)[:, 1]

		test_accuracy = accuracy_score(y_test, y_pred)
		test_recall = recall_score(y_test, y_pred)
		test_f1 = f1_score(y_test, y_pred)
		test_auroc = roc_auc_score(y_test, y_proba)
		test_aucpr = average_precision_score(y_test, y_proba)
		cm = confusion_matrix(y_test, y_pred)

		auc_rows.append(
			{
				"Model Name": model_name,
				"AUCPR": test_aucpr,
				"AUROC": test_auroc,
			}
		)

		metrics_rows.append(
			{
				"Model Name": model_name,
				"Accuracy": test_accuracy,
				"Recall": test_recall,
				"F1-Score": test_f1,
			}
		)

		confusion_details.append(
			"\n".join(
				[
					f"Model: {model_name}",
					"Confusion Matrix (rows=true class [0,1], cols=predicted class [0,1]):",
					str(cm),
					f"TN={cm[0, 0]}, FP={cm[0, 1]}, FN={cm[1, 0]}, TP={cm[1, 1]}",
					"",
				]
			)
		)

	return (
		pd.DataFrame(cv_rows),
		pd.DataFrame(auc_rows),
		pd.DataFrame(metrics_rows),
		"\n".join(confusion_details),
	)


def process_single_run(run_id: int) -> bool:
	"""Process one run directory and return True when completed successfully."""
	run_dir = BASE_RUN_DIR / f"Run_{run_id}"
	train_path = run_dir / "synthetic_data.csv"
	output_dir = run_dir

	if not run_dir.exists() or not train_path.exists():
		print(
			f"Warning: Skipping Run_{run_id} because required data is missing. "
			f"Expected file: {train_path}"
		)
		return False

	output_dir.mkdir(parents=True, exist_ok=True)
	print(f"\n========== Processing Run_{run_id} ==========")

	try:
		x_train, y_train, x_test, y_test = load_data(train_path, TEST_PATH, TARGET_COLUMN)

		# Keep test feature columns aligned with training columns.
		x_test = x_test.reindex(columns=x_train.columns)
		if x_test.isnull().any().any():
			missing_cols = [col for col in x_test.columns if x_test[col].isnull().all()]
			raise ValueError(
				"Test feature set is missing one or more columns found in training data. "
				f"Missing columns: {missing_cols}"
			)

		y_train_encoded, y_test_encoded, label_encoder = encode_target(y_train, y_test)
		print(f"Encoded target classes: {list(label_encoder.classes_)} -> [0, 1]")

		models = build_models()
		cv_df, table_auc_df, table_metrics_df, confusion_text = evaluate_models(
			models,
			x_train,
			y_train_encoded,
			x_test,
			y_test_encoded,
		)

		table_auc_path = output_dir / "Table_AUC.csv"
		table_metrics_path = output_dir / "Table_Metrics.csv"
		confusion_path = output_dir / "Confusion_Matrices.txt"

		table_auc_df.to_csv(table_auc_path, index=False)
		table_metrics_df.to_csv(table_metrics_path, index=False)
		confusion_path.write_text(confusion_text, encoding="utf-8")

		print("\n=== 10-Fold Stratified CV (Training Set, Mean Scores) ===")
		print(cv_df.to_string(index=False))
		print("\n=== Test Set Evaluation Saved ===")
		print(f"AUC table: {table_auc_path}")
		print(f"Metrics table: {table_metrics_path}")
		print(f"Confusion matrices: {confusion_path}")

		return True

	except FileNotFoundError as exc:
		print(f"Warning: Skipping Run_{run_id}. {exc}")
		return False

	except Exception as exc:
		print(f"Warning: Run_{run_id} failed with error: {exc}")
		return False


def main() -> None:
	processed_runs: List[int] = []
	skipped_runs: List[int] = []

	for run_id in RUN_IDS:
		if process_single_run(run_id):
			processed_runs.append(run_id)
		else:
			skipped_runs.append(run_id)

	print("\n========== Global Summary ==========")
	print(f"Successfully processed runs: {processed_runs if processed_runs else 'None'}")
	print(f"Skipped/failed runs: {skipped_runs if skipped_runs else 'None'}")


if __name__ == "__main__":
	main()


## ***Aggregate_Result***

In [14]:
import argparse
from pathlib import Path
from typing import List, Tuple

import pandas as pd


RUN_NAMES = [f"Run_{i}" for i in range(1, 6)]
AUC_FILE = "Table_AUC.csv"
METRICS_FILE = "Table_Metrics.csv"
AUC_METRICS = ["AUROC", "AUCPR"]
CORE_METRICS = ["Accuracy", "Recall", "F1-Score"]


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]
    return df


def _find_model_column(df: pd.DataFrame) -> str:
    candidates = ["Model Name", "Model", "Classifier"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(
        f"Could not find a model column. Expected one of: {candidates}. Found: {list(df.columns)}"
    )


def _load_csv_if_exists(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")
    return _normalize_columns(pd.read_csv(file_path))


def collect_run_tables(base_dir: Path) -> Tuple[List[pd.DataFrame], List[pd.DataFrame]]:
    auc_tables: List[pd.DataFrame] = []
    metrics_tables: List[pd.DataFrame] = []

    for run_name in RUN_NAMES:
        run_dir = base_dir / run_name
        if not run_dir.exists():
            print(f"[SKIP] Missing folder: {run_dir}")
            continue

        auc_path = run_dir / AUC_FILE
        metrics_path = run_dir / METRICS_FILE

        try:
            auc_df = _load_csv_if_exists(auc_path)
            auc_df["_Run"] = run_name
            auc_tables.append(auc_df)
        except Exception as exc:
            print(f"[SKIP] {run_name} - AUC table issue: {exc}")

        try:
            metrics_df = _load_csv_if_exists(metrics_path)
            metrics_df["_Run"] = run_name
            metrics_tables.append(metrics_df)
        except Exception as exc:
            print(f"[SKIP] {run_name} - Metrics table issue: {exc}")

    return auc_tables, metrics_tables


def aggregate_table(
    tables: List[pd.DataFrame],
    metric_columns: List[str],
) -> pd.DataFrame:
    if not tables:
        return pd.DataFrame()

    combined = pd.concat(tables, ignore_index=True, sort=False)
    combined = _normalize_columns(combined)

    model_col = _find_model_column(combined)
    available_metrics = [col for col in metric_columns if col in combined.columns]

    if not available_metrics:
        print(
            f"[WARN] None of requested metrics {metric_columns} found. Available: {list(combined.columns)}"
        )
        return pd.DataFrame()

    grouped = combined.groupby(model_col, dropna=False)[available_metrics]
    mean_df = grouped.mean(numeric_only=True)
    std_df = grouped.std(numeric_only=True).fillna(0.0)

    formatted = pd.DataFrame(index=mean_df.index)
    for metric in available_metrics:
        formatted[metric] = [
            f"{mean_df.loc[idx, metric]:.3f} ± {std_df.loc[idx, metric]:.3f}"
            for idx in mean_df.index
        ]

    formatted = formatted.reset_index()
    return formatted


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Aggregate experiment results from Run_1 to Run_5."
    )
    parser.add_argument(
        "--base-dir",
        type=str,
        default="/content/drive/MyDrive/PATE-GAN",
        help="Base directory that contains Run_1 ... Run_5 folders.",
    )
    # Jupyter/Colab kernels append extra CLI args (e.g., -f <kernel.json>).
    # parse_known_args keeps normal CLI behavior while safely ignoring unknowns.
    args, unknown_args = parser.parse_known_args()
    if unknown_args:
        print(f"[INFO] Ignoring unknown arguments: {unknown_args}")

    base_dir = Path(args.base_dir)
    if not base_dir.exists():
        raise FileNotFoundError(f"Base directory does not exist: {base_dir}")

    auc_tables, metrics_tables = collect_run_tables(base_dir)

    final_auc = aggregate_table(auc_tables, AUC_METRICS)
    final_metrics = aggregate_table(metrics_tables, CORE_METRICS)

    out_dir = base_dir / "Aggregation_Summary"
    out_dir.mkdir(parents=True, exist_ok=True)

    auc_out_path = out_dir / "Final_Aggregated_AUC.csv"
    metrics_out_path = out_dir / "Final_Aggregated_Metrics.csv"

    if not final_auc.empty:
        final_auc.to_csv(auc_out_path, index=False)
        print(f"[OK] Saved: {auc_out_path}")
    else:
        print("[WARN] No aggregated AUC table generated.")

    if not final_metrics.empty:
        final_metrics.to_csv(metrics_out_path, index=False)
        print(f"[OK] Saved: {metrics_out_path}")
    else:
        print("[WARN] No aggregated metrics table generated.")


if __name__ == "__main__":
    main()


[INFO] Ignoring unknown arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-4c5f5ec7-1f44-44e6-be16-4fbb57af73d6.json']
[OK] Saved: /content/drive/MyDrive/PATE-GAN/Aggregation_Summary/Final_Aggregated_AUC.csv
[OK] Saved: /content/drive/MyDrive/PATE-GAN/Aggregation_Summary/Final_Aggregated_Metrics.csv


In [16]:
df_results = pd.read_csv("/content/drive/MyDrive/PATE-GAN/Aggregation_Summary/Final_Aggregated_Metrics.csv")
df_results

,Model Name,Accuracy,Recall,F1-Score
0,AdaBoost,0.779 ± 0.017,0.471 ± 0.222,0.474 ± 0.152
1,KNN,0.760 ± 0.004,0.423 ± 0.111,0.448 ± 0.063
2,Logistic Regression,0.779 ± 0.014,0.468 ± 0.153,0.490 ± 0.075
3,XGBoost,0.783 ± 0.005,0.414 ± 0.198,0.449 ± 0.131


In [17]:
df_results_auc = pd.read_csv("/content/drive/MyDrive/PATE-GAN/Aggregation_Summary/Final_Aggregated_AUC.csv")
df_results_auc

,Model Name,AUROC,AUCPR
0,AdaBoost,0.817 ± 0.015,0.544 ± 0.028
1,KNN,0.767 ± 0.025,0.460 ± 0.032
2,Logistic Regression,0.819 ± 0.007,0.521 ± 0.019
3,XGBoost,0.822 ± 0.009,0.542 ± 0.025
